In [1]:
import probablepeople as pp

In [ ]:
import os
import sys

# 1. Point directly to your installed JDK 17 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Ensure PySpark uses the active Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Now import and start Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "24") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .getOrCreate()
print("Spark initialized successfully!")

Spark initialized successfully!


USER_NAME

In [39]:
df = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_user.json")

In [40]:
df_filtered_ch = df.filter(
    # Must contain at least one English letter OR CJK character
    (df["name"].rlike(r"[a-zA-Z]") | df["name"].rlike(r"[\u4E00-\u9FFF\u3040-\u309F\u30A0-\u30FF]"))
    &
    # AND must NOT contain numbers or junk symbols
    (~df["name"].rlike(r"[0-9@#$%^&*()_+=\[\]{};:\"\\|<>/?]"))
)
df_filtered_ch.cache()

DataFrame[average_stars: double, compliment_cool: bigint, compliment_cute: bigint, compliment_funny: bigint, compliment_hot: bigint, compliment_list: bigint, compliment_more: bigint, compliment_note: bigint, compliment_photos: bigint, compliment_plain: bigint, compliment_profile: bigint, compliment_writer: bigint, cool: bigint, elite: string, fans: bigint, friends: string, funny: bigint, name: string, review_count: bigint, useful: bigint, user_id: string, yelping_since: string]

In [41]:
list_names=[]
for row in df_filtered_ch.select("name").distinct().collect():
    parsed = pp.parse(row.name)
    list_names.append(parsed) #parsed.split(",")
   #
   #  list_names = " ".join(parsed)

In [42]:
unauthorized_username={"CorporationNameAndCompany","CorporationName","CorporationNameOrganization","CorporationLegalType","CorporationCommitteeType","CorporationNameBranchType","CorporationNameBranchIdentifier"}

In [43]:
tap=[]
des=set()
#print(type(des))
for row in list_names:
  t=""
  count=0
  for i,j in row:
      if j not in unauthorized_username:
        t=t+""+i
        count=count+1
        des.add(j)
#        print(i,"- > ",j,"->---->",count)
 # print("------------------------",t) 
      
  tap.append(t) 

In [ ]:
df_filtered_ch_2=df_filtered_ch.filter(df_filtered_ch["name"].isin(tap))
df_filtered_ch_2.cache()

In [ ]:
#from pyspark.sql.functions import broadcast

# Create a small DataFrame from your list/set
#tap_df = spark.createDataFrame([(x,) for x in tap], ["tap_name"]).repartition(16) #tap_df = spark.createDataFrame([(x,) for x in tap], ["tap_name"]).repartition(16)

# Broadcast join is 10x-50x faster than .isin() over large lists
#df_filtered_ch_2 = df_filtered_ch.repartition(32, "name").join(
 #   broadcast(tap_df),
  #  df_filtered_ch["name"] == tap_df["tap_name"],
   # "right"
#).drop("tap_name")



In [ ]:
df_filtered_ch_2.select(df_filtered_ch_2.name).show(n=1000)

In [ ]:
df.select(df.name).show(n=1000)

review_count

In [ ]:
# review count df.select(df["review_count"]).filter(f.col("review_count").try_cast("int") >=0)

Join_Date

In [ ]:
df_filtered = df.filter(
    F.to_date(F.col("yelping_since")).between("2004-10-01", F.current_date())
).withColumn("Join_Date", F.to_date(F.col("yelping_since")))

In [ ]:
df_filt = df.filter((f.col("useful").try_cast("int") >=0)
                     & (f.col("funny").try_cast("int") >=0) 
                     & (f.col("cool").try_cast("int") >=0) 
                     & (f.col("fans").try_cast("int") >=0)
                     & (f.col("compliment_hot").try_cast("int") >=0)
                     & (f.col("compliment_more").try_cast("int") >=0) 
                     & (f.col("compliment_profile").try_cast("int") >=0) 
                     & (f.col("compliment_cute").try_cast("int") >=0)
                     & (f.col("compliment_list").try_cast("int") >=0)
                     & (f.col("compliment_note").try_cast("int") >=0) 
                     & (f.col("compliment_plain").try_cast("int") >=0) 
                     & (f.col("compliment_cool").try_cast("int") >=0)
                     & (f.col("compliment_funny").try_cast("int") >=0)
                     & (f.col("compliment_writer").try_cast("int") >=0) 
                     & (f.col("compliment_photos").try_cast("int") >=0) 
                     & (f.col("average_stars").try_cast("float") >=0))